# 03 — Entities and Animals: Audit and Re-estimation

MallWorld audit (2026-10)

---

## Scope

This notebook re-derives the entity claims of:
- *Entity Ecology in MallWorld Dreams* (archived notebook 05);
- the thesis, §4 "Entity Behavioral Patterns" and §5 "Animal Correspondences" (archived notebook 09, Phases 3–4 and 8);
- *Vertical World Structure* §§3.7–3.9 (archived notebook 07).

**Unit of analysis.** Entities are recorded only inside interactions, so one entity can appear several times. Rates of "entity type at a location" use unique (dream, location, entity type) triples. Inference is clustered on the dream.

**Definitional links.** Some entity types carry their behaviour in their labels. The schema comments define `guide` as "someone helpful", `threat` as "hostile entity" and `watcher` as "observer entity"; the extraction model saw the labels themselves. Associations between these types and demeanor, or between threats and threatening atmospheres, are therefore partly built into the coding. They describe the coder's consistency, not the dream world.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

L = mw.load_locations()
E = mw.load_entities()
I = mw.load_interactions()
L["neg"] = (L["valence"] == -1).astype(float).where(L["valence"].notna())
L["level_cat"] = pd.cut(L["vertical_level"], [-3, -0.5, 0.5, 3], labels=["underground", "ground", "elevated"])
EL = E.merge(L[["post_id", "location_id", "valence", "neg", "vertical_level", "level_cat", "location_type", "atmosphere"]],
             on=["post_id", "location_id"], how="left")
assert len(EL) == len(E)
print(f"Entity involvements: {len(E):,} in {E['post_id'].nunique():,} dreams ({100 * E['post_id'].nunique() / len(primary):.1f}% of primary dreams)")
print(E["entity_type"].value_counts().to_string())

def cluster_prop(df, y, cluster="post_id"):
    d = df.dropna(subset=[y]); n = len(d)
    if n == 0:
        return np.nan, np.nan, np.nan, 0, 0
    p = d[y].mean(); tot = d.groupby(cluster)[y].agg(["sum", "size"]); C = len(tot)
    se = np.sqrt((((tot["sum"] - p * tot["size"]) ** 2).sum() * C / max(C - 1, 1)) / n ** 2)
    return p, max(p - 1.96 * se, 0), min(p + 1.96 * se, 1), n, C

def fmt(df, y):
    p, lo, hi, n, C = cluster_prop(df, y)
    return f"{100 * p:.1f}% [{100 * lo:.1f}–{100 * hi:.1f}] (n = {n}, dreams = {C})"

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


Entity involvements: 3,696 in 1,275 dreams (66.5% of primary dreams)
entity_type
stranger         911
crowd            808
authority        469
threat           306
known_person     245
other            236
family_member    224
creature         194
friend            96
child             72
deceased          56
guide             32
watcher           14
shadow            11
coworker          11
mannequin          5
faceless           3
none               3


## E1 — Census, and dream-level effects that are narrative length

**Old (entity ecology report).**
- "66.4% of dreams contain entities".
- "Dreams with threats averaged 5.35 locations vs 4.43" (t = 3.00).
- "Threatening atmospheres 35.3% vs 6.7% in threat-absent dreams".
- "Family-present dreams more threatening (15.3% vs 9.1%), the family paradox".

In [2]:
per_dream = E.groupby("post_id")["entity_type"].apply(set)
P = primary.set_index("post_id").copy()
for t in ["threat", "family_member", "authority", "deceased", "creature"]:
    P[f"has_{t}"] = P.index.map(lambda pid: t in per_dream.get(pid, set()))
print(f"Dreams with >= 1 entity: {100 * P.index.isin(per_dream.index).mean():.1f}%")
for t in ["threat", "family_member", "authority"]:
    a, b = P[P[f"has_{t}"]], P[~P[f"has_{t}"]]
    print(f"{t:14s}: words median {a['words'].median():.0f} vs {b['words'].median():.0f}; locations mean {a['n_locations'].mean():.2f} vs {b['n_locations'].mean():.2f}")
# Archived metric: per-dream share of locations coded threatening (all locations in the denominator)
Ld = L.copy()
Ld["threatening"] = (Ld["atmosphere"] == "threatening").astype(float)
share = Ld.groupby("post_id")["threatening"].mean()
P["pct_thr"] = share.reindex(P.index).fillna(0)
P["any_entity"] = P.index.isin(per_dream.index)
for t in ["threat", "family_member", "authority", "creature"]:
    a, b = P.loc[P[f"has_{t}"], "pct_thr"], P.loc[~P[f"has_{t}"], "pct_thr"]
    b2 = P.loc[~P[f"has_{t}"] & P["any_entity"], "pct_thr"]
    print(f"Archived metric, {t:14s}: {100 * a.mean():.1f}% vs {100 * b.mean():.1f}% (vs {100 * b2.mean():.1f}% in dreams with other entities only); n = {len(a)} dreams")
# Location level, length-adjusted, clustered by dream
for t in ["threat", "family_member", "authority"]:
    Ld[f"has_{t}"] = Ld["post_id"].map(P[f"has_{t}"]).astype(int)
    r = mw.clustered_logit(Ld, f"threatening ~ has_{t} + log_words")
    tt = mw.odds_ratio_table(r, [f"has_{t}"]).iloc[0]
    print(f"Location coded threatening, dream has {t}: {Ld.loc[Ld[f'has_{t}'] == 1, 'threatening'].mean() * 100:.1f}% vs {Ld.loc[Ld[f'has_{t}'] == 0, 'threatening'].mean() * 100:.1f}%; "
          f"length-adjusted OR {tt.OR:.2f} [{tt.lo:.2f}-{tt.hi:.2f}], p = {tt.p:.2g}")
Ld["any_entity"] = Ld["post_id"].map(P["any_entity"]).astype(int)
Le = Ld[Ld["any_entity"] == 1]
r = mw.clustered_logit(Le, "threatening ~ has_family_member + log_words")
tt = mw.odds_ratio_table(r, ["has_family_member"]).iloc[0]
print(f"Family member, among dreams with any entity: length-adjusted OR {tt.OR:.2f} [{tt.lo:.2f}-{tt.hi:.2f}], p = {tt.p:.2g}")
# Location level: family present AT the location vs other locations of dreams with entities
fam_locs = set(E.loc[E["entity_type"] == "family_member", ["post_id", "location_id"]].itertuples(index=False, name=None))
Le = Le.assign(fam_here=[int((p_, l_) in fam_locs) for p_, l_ in zip(Le["post_id"], Le["location_id"])])
r = mw.clustered_logit(Le.dropna(subset=["neg"]), "neg ~ fam_here + log_words")
tt = mw.odds_ratio_table(r, ["fam_here"]).iloc[0]
print(f"Negative atmosphere where a family member is present (vs other coded locations in dreams with entities): "
      f"{100 * Le.loc[Le['fam_here'] == 1, 'neg'].mean():.1f}% vs {100 * Le.loc[Le['fam_here'] == 0, 'neg'].mean():.1f}%, OR {tt.OR:.2f} [{tt.lo:.2f}-{tt.hi:.2f}], p = {tt.p:.2g}")

Dreams with >= 1 entity: 66.5%
threat        : words median 232 vs 143; locations mean 5.39 vs 4.44
family_member : words median 237 vs 144; locations mean 5.16 vs 4.48
authority     : words median 278 vs 139; locations mean 5.95 vs 4.29
Archived metric, threat        : 35.3% vs 6.7% (vs 7.9% in dreams with other entities only); n = 187 dreams
Archived metric, family_member : 15.3% vs 9.0% (vs 11.5% in dreams with other entities only); n = 138 dreams
Archived metric, authority     : 13.0% vs 8.9% (vs 11.6% in dreams with other entities only); n = 280 dreams
Archived metric, creature      : 13.7% vs 9.2% (vs 11.7% in dreams with other entities only); n = 130 dreams


Location coded threatening, dream has threat: 24.8% vs 6.0%; length-adjusted OR 5.14 [4.11-6.43], p = 1.1e-46
Location coded threatening, dream has family_member: 10.4% vs 8.0%; length-adjusted OR 1.28 [0.92-1.77], p = 0.14
Location coded threatening, dream has authority: 10.7% vs 7.6%; length-adjusted OR 1.34 [1.06-1.68], p = 0.014
Family member, among dreams with any entity: length-adjusted OR 1.01 [0.73-1.40], p = 0.93
Negative atmosphere where a family member is present (vs other coded locations in dreams with entities): 68.6% vs 67.2%, OR 1.06 [0.73-1.53], p = 0.77


### Findings — E1

**Reproduced.** The archived dream-level figures reproduce on the primary population:
- 66.5% of dreams contain at least one entity (archived 66.4%);
- dreams with a threat have more locations (5.39 vs 4.44);
- threat dreams have a higher share of threatening locations (35.3% vs 6.7%);
- family dreams have a higher share of threatening locations (15.3% vs 9.0%).

**What they measure.** Dreams with entities are longer: the median is 232 words for dreams with a threat, against 143 for dreams without one. Longer reports have more locations and more coded atmospheres.

- **Threat → threatening atmosphere.** The location-level association survives length adjustment: OR 5.14 (95% CI 4.11–6.43). It is close to definitional, because a location where a hostile entity acts is coded as threatening. It shows that the coder is consistent. It is not a finding about the dream world.
- **The "family paradox" is withdrawn.**
  - The archived contrast compared family dreams with all other dreams, most of which are short and contain no entity.
  - Against dreams with other entities, the archived metric is 15.3% vs 11.5%.
  - After length adjustment the OR is 1.28 (0.92–1.77, p = 0.14), and among dreams with any entity it is 1.01.
  - Where a family member actually appears, the location is negative in 68.6% of cases, against 67.2% for other coded locations in dreams with entities (OR 1.06, p = 0.77).
- **Authority.** Dreams with an authority figure have slightly more threatening locations after length adjustment (OR 1.34, 1.06–1.68, p = 0.014).

## E2 — Entity types and atmosphere

**Old.**
- "χ² = 792, df = 99, p = 10⁻¹⁰⁸; threat–threatening z = +14.86 (71% vs 19%)".
- Crowds–chaotic z = +7.05; authority–oppressive z = +4.86.
- Deceased: "37% welcoming or peaceful vs 8% baseline".

The deceased contrast was pre-registered and is tested in notebook 07 (P3).

In [3]:
U = EL.drop_duplicates(["post_id", "location_id", "entity_type"]).dropna(subset=["valence"])
tab = pd.crosstab(U["entity_type"], U["atmosphere"])
print(f"Unique (dream, location, type) with atmosphere: {len(U)}; archived-style table {tab.shape}; {mw.expected_count_report(tab)}")
rows = []
for t, g in U.groupby("entity_type"):
    if len(g) < 25:
        continue
    p, lo, hi, n, C = cluster_prop(g.assign(thr=(g["atmosphere"] == "threatening").astype(float)), "thr")
    pn = cluster_prop(g, "neg")
    rows.append({"entity type": t, "n": n, "dreams": C, "threatening %": 100 * p, "negative %": 100 * pn[0],
                 "negative 95% CI": f"{100 * pn[1]:.0f}–{100 * pn[2]:.0f}"})
print(pd.DataFrame(rows).set_index("entity type").sort_values("negative %", ascending=False).round(1).to_string())

Unique (dream, location, type) with atmosphere: 2194; archived-style table (18, 9); min expected = 0.05; cells with expected < 5: 53%
                 n  dreams  threatening %  negative % negative 95% CI
entity type                                                          
threat         219     166           70.3        95.4           93–98
creature       100      86           43.0        84.0           76–92
other          145     121           33.1        83.4           77–90
authority      279     210           29.4        79.9           75–85
child           51      42           21.6        76.5           64–89
crowd          547     420           17.4        72.6           69–76
known_person   111      86           12.6        72.1           63–81
family_member  121     102           17.4        68.6           61–77
stranger       499     370           14.8        66.3           62–71
friend          50      45           18.0        58.0           44–72


### Findings — E2

**The archived χ² (792, df = 99) is not a valid test.** It counts entity involvements, which are nested in dreams, and the table has a majority of sparse cells. Our reconstruction has 53% of cells with an expected count below 5.

**Corrected rates** (unique dream × location × entity type; CIs clustered by dream):

| Entity type | Locations | Dreams | Negative atmosphere (95% CI) |
|---|---|---|---|
| threat | 219 | 166 | 95.4% (93–98) |
| creature | 100 | 86 | 84.0% (76–92) |
| authority | 279 | 210 | 79.9% (75–85) |
| crowd | 547 | 420 | 72.6% (69–76) |
| family member | 121 | 102 | 68.6% (61–77) |
| stranger | 499 | 370 | 66.3% (62–71) |
| friend | 50 | 45 | 58.0% (44–72) |

**Interpretation.**
- Threats, creatures and authority figures appear in negative scenes; friends, strangers and family appear in less negative ones.
- The threat row is definitional.
- The other rows fit the framework's claim that what surrounds a person corresponds to their state (*Heaven and Hell* §§173–176). They also fit the ordinary reading that a hostile guard or an animal makes a scene unpleasant. This table cannot tell the two apart.
- The deceased contrast was registered in advance and is tested in notebook 07 (P3, a hit).

## E3 — "Entity autonomy": behaviour by level and atmosphere (the join bug)

**Old (thesis §4.3 and FINDINGS_LOG E1–E7).**
- "Guides maintained 0% hostility at all vertical levels, threats 92–94% hostility regardless of atmosphere".
- "Authority functions identical at all levels (p = 0.78)".
- "Creature hostility ~35% at ALL atmospheres".

All of these were computed on the join that attached each entity to locations from other dreams (notebook 01, D6).

In [4]:
HOSTILE = ["hostile", "threatening"]
mw.check_values("demeanor", HOSTILE)
EL["hostile_"] = EL["demeanor"].isin(HOSTILE).astype(float)
print("Hostile demeanor by entity type (definitional for guide/threat):")
for t in ["guide", "threat", "authority", "creature", "stranger", "crowd", "deceased"]:
    print(f"  {t:10s} {fmt(EL[EL['entity_type'] == t], 'hostile_')}")

# The archived 'invariance' table, recomputed with the correct key
Ex = mw.load_entities("extractable"); Lx = mw.load_locations("extractable")
Lx["level_cat"] = pd.cut(Lx["vertical_level"], [-3, -0.5, 0.5, 3], labels=["underground", "ground", "elevated"])
Ex["hostile_"] = Ex["demeanor"].isin(HOSTILE).astype(float)
wrong = Ex.merge(Lx[["location_id", "level_cat"]], on="location_id")
right = Ex.merge(Lx[["post_id", "location_id", "level_cat"]], on=["post_id", "location_id"])
print("\nHostile rate by level, all entities (thesis population):")
print("  wrong join :", wrong.groupby("level_cat", observed=True)["hostile_"].mean().mul(100).round(1).to_dict(), f"(rows {len(wrong):,})")
print("  correct key:", right.groupby("level_cat", observed=True)["hostile_"].mean().mul(100).round(1).to_dict(), f"(rows {len(right):,})")

print("\nPrimary population, correct key, by level:")
for t in ["threat", "authority", "creature", "stranger"]:
    sub = EL[(EL["entity_type"] == t) & EL["level_cat"].notna()]
    print(f"  {t:10s}", {lvl: fmt(g, "hostile_") for lvl, g in sub.groupby("level_cat", observed=True)})
AN = EL[(EL["entity_type"] == "authority") & (EL["authority_nature"] != "not_mentioned") & EL["level_cat"].notna()]
ct = pd.crosstab(AN["authority_nature"], AN["level_cat"])
chi, pv, dof, _ = chi2_contingency(ct)
print(f"\nAuthority nature x level (n = {len(AN)} involvements, {AN['post_id'].nunique()} dreams): chi2 = {chi:.1f}, df = {dof}, p = {pv:.2g}; {mw.expected_count_report(ct)}")
print(ct.to_string())
# The archived notebook's own stated prediction (exploratory pattern 3): guiding/observing above, blocking/pursuing/punitive below
AN = AN.assign(benign=AN["authority_nature"].isin(["guiding", "observing"]).astype(float))
print("Guiding or observing share of authority functions:", {lvl: fmt(g, "benign") for lvl, g in AN.groupby("level_cat", observed=True)})
ANd = AN[AN["level_cat"] != "ground"].assign(elev=lambda d: (d["level_cat"] == "elevated").astype(int))
r = mw.clustered_logit(ANd, "benign ~ elev")
print("  elevated vs underground:", mw.odds_ratio_table(r, ["elev"]).round(3).to_dict("index"))

# 'Threats present equally (8.6%) across all atmospheres' and 'guides 5.2-5.3% at all levels'
Lt = L.dropna(subset=["valence"]).copy()
thr_locs = set(E.loc[E["entity_type"] == "threat", ["post_id", "location_id"]].itertuples(index=False, name=None))
Lt["threat_here"] = [float((p_, l_) in thr_locs) for p_, l_ in zip(Lt["post_id"], Lt["location_id"])]
print("\nShare of locations with a threat entity, by atmosphere valence:", {int(v): fmt(g, "threat_here") for v, g in Lt.groupby("valence")})
gd = EL[EL["entity_type"] == "guide"]
print(f"Guide involvements with a vertical level: {int(gd['level_cat'].notna().sum())} of {len(gd)} "
      f"({gd.loc[gd['level_cat'].notna(), 'post_id'].nunique()} dreams):", gd["level_cat"].value_counts().to_dict())

Hostile demeanor by entity type (definitional for guide/threat):
  guide      0.0% [0.0–0.0] (n = 32, dreams = 17)
  threat     93.8% [90.0–97.6] (n = 306, dreams = 187)
  authority  33.3% [27.6–38.9] (n = 469, dreams = 280)
  creature   39.2% [30.4–48.0] (n = 194, dreams = 130)
  stranger   8.1% [6.1–10.2] (n = 911, dreams = 528)
  crowd      8.7% [6.4–10.9] (n = 808, dreams = 566)
  deceased   3.6% [0.0–10.6] (n = 56, dreams = 33)



Hostile rate by level, all entities (thesis population):
  wrong join : {'underground': 19.7, 'ground': 19.5, 'elevated': 19.6} (rows 7,158,006)
  correct key: {'underground': 21.8, 'ground': 20.2, 'elevated': 19.5} (rows 4,208)

Primary population, correct key, by level:
  threat     {'underground': '95.5% [86.4–100.0] (n = 22, dreams = 18)', 'ground': '90.9% [81.8–100.0] (n = 55, dreams = 39)', 'elevated': '88.5% [72.4–100.0] (n = 26, dreams = 16)'}
  authority  {'underground': '23.8% [1.7–45.9] (n = 21, dreams = 18)', 'ground': '40.0% [26.0–54.0] (n = 75, dreams = 50)', 'elevated': '28.3% [14.9–41.6] (n = 46, dreams = 38)'}
  creature   {'underground': '41.7% [15.3–68.0] (n = 24, dreams = 15)', 'ground': '37.0% [16.4–57.7] (n = 27, dreams = 22)', 'elevated': '44.4% [16.7–72.2] (n = 18, dreams = 15)'}
  stranger   {'underground': '1.6% [0.0–4.9] (n = 61, dreams = 43)', 'ground': '4.5% [0.6–8.4] (n = 133, dreams = 84)', 'elevated': '7.9% [1.6–14.1] (n = 89, dreams = 65)'}

Authority 


Share of locations with a threat entity, by atmosphere valence: {-1: '7.6% [6.4–8.8] (n = 2748, dreams = 1100)', 0: '0.5% [0.0–0.9] (n = 859, dreams = 460)', 1: '1.1% [0.2–2.0] (n = 538, dreams = 345)'}
Guide involvements with a vertical level: 18 of 32 (10 dreams): {'elevated': 8, 'underground': 5, 'ground': 5}


### Findings — E3

**The "entity autonomy" claims rest on the join bug and on definitions.**

- **Guides 0% hostile, threats 93.8% hostile.** These follow from the labels. A guide coded as hostile, or a threat coded as friendly, would be a coding contradiction. Neither figure says anything about autonomy.
- **Flat hostility across levels.**
  - Under the archived join, the hostile rate is 19.7 / 19.5 / 19.6% (underground / ground / elevated) over 7,158,006 rows. It is flat by construction, because every entity was attached to the matching location number in every dream.
  - With the correct key, the 4,208 involvements give 21.8 / 20.2 / 19.5%.
  - Within types in the primary population, hostility by level is:
    - threats 95.5 / 90.9 / 88.5%;
    - authority 23.8 / 40.0 / 28.3%;
    - creatures 41.7 / 37.0 / 44.4%.
  - With 15–50 dreams per cell, the CIs are up to 53 points wide. These data can show neither invariance (that would need an equivalence test with a useful margin) nor variation. **Underdetermined.**
- **Authority function by level.**
  - The archived notebook stated a prediction before this test: guiding and observing above, blocking, pursuing and punishing below. Its own correctly joined run gave χ²(8) = 13.42, p = 0.098, with residuals *opposite* to that prediction.
  - The thesis instead reported the join-bug figure ("identical at all levels, p = 0.78") and labelled it as support for entity autonomy.
  - On the primary population with the correct key: χ²(8) = 9.2, p = 0.33 (27% sparse cells; 134 involvements in 93 dreams).
  - The guiding-or-observing share is 55.0% underground, 43.1% at ground level and 35.7% elevated (elevated vs underground OR 0.46, 0.12–1.71, p = 0.24).
  - **Verdict: miss in direction, not significant (underpowered).** A miss relabelled as a hit is withdrawn.
- **"Threats present equally (8.6%) at all atmospheres."** This was join-bug output. A threat entity is present at 7.6% of negative locations, 0.5% of neutral ones and 1.1% of positive ones.
- **Guides.** There are 32 involvements in 17 dreams; 18 of them have a vertical level, in 10 dreams. "Guides 5.2–5.3% at every level" cannot be estimated from these data.

## E4 — Entity types by vertical level (vertical report §3.7; the χ² = 41.13)

**Old.**
- "Entity type × vertical level χ² = 41.13, df = 22, p = 0.008".
- "Creatures 11.6% of entities underground vs 5.4% elsewhere; authority 14.9% elevated vs 9.5% underground".
- Archived notebook 09 called H3 ("creatures below, authority above") pre-registered and supported.

This statistic was also attributed, in the earlier `CLAUDE.md`, to the **NDE** entity-function claim.

In [5]:
UL = EL.drop_duplicates(["post_id", "location_id", "entity_type"]).dropna(subset=["level_cat"])
print(f"Unique (dream, location, type) with a level: {len(UL)} in {UL['post_id'].nunique()} dreams")
for t in ["creature", "authority", "threat"]:
    d = UL.assign(y=(UL["entity_type"] == t).astype(float))
    print(f"{t:10s} share by level:", {lvl: fmt(g, "y") for lvl, g in d.groupby("level_cat", observed=True)})
    d["under"] = (d["vertical_level"] < 0).astype(int); d["elev"] = (d["vertical_level"] > 0).astype(int)
    r = mw.clustered_logit(d, "y ~ under + elev")
    print("           ", mw.odds_ratio_table(r, ["under", "elev"]).round(3).to_dict("index"))
# Reproduce the archived statistic: involvements (not unique), thesis population, types with >= 10 at defined levels
Ex_l = Ex.merge(Lx[["post_id", "location_id", "level_cat"]], on=["post_id", "location_id"]).dropna(subset=["level_cat"])
ct_old = pd.crosstab(Ex_l["entity_type"], Ex_l["level_cat"])
ct_old = ct_old[ct_old.sum(axis=1) >= 10]
chi, pv, dof, _ = chi2_contingency(ct_old)
print(f"\nArchived construction: {int(ct_old.values.sum())} involvements, chi2 = {chi:.2f}, df = {dof}, p = {pv:.4f}; {mw.expected_count_report(ct_old)}")
print(f"  dreams contributing: {Ex_l['post_id'].nunique()}; involvements per dream up to {Ex_l.groupby('post_id').size().max()}")

Unique (dream, location, type) with a level: 953 in 478 dreams
creature   share by level: {'underground': '8.3% [4.7–12.0] (n = 180, dreams = 120)', 'ground': '4.6% [2.7–6.4] (n = 504, dreams = 270)', 'elevated': '5.9% [3.0–8.9] (n = 269, dreams = 180)'}
            {'under': {'OR': 1.901, 'lo': 1.0, 'hi': 3.613, 'p': 0.05}, 'elev': {'OR': 1.323, 'lo': 0.667, 'hi': 2.624, 'p': 0.424}}
authority  share by level: {'underground': '10.6% [6.0–15.1] (n = 180, dreams = 120)', 'ground': '11.3% [8.4–14.2] (n = 504, dreams = 270)', 'elevated': '14.5% [10.5–18.5] (n = 269, dreams = 180)'}
            {'under': {'OR': 0.925, 'lo': 0.533, 'hi': 1.607, 'p': 0.783}, 'elev': {'OR': 1.33, 'lo': 0.875, 'hi': 2.021, 'p': 0.182}}
threat     share by level: {'underground': '10.6% [6.0–15.1] (n = 180, dreams = 120)', 'ground': '8.5% [5.9–11.2] (n = 504, dreams = 270)', 'elevated': '7.4% [3.7–11.2] (n = 269, dreams = 180)'}
            {'under': {'OR': 1.265, 'lo': 0.713, 'hi': 2.244, 'p': 0.421}, 'elev': {

### Findings — E4

**Source of χ² = 41.13.** The archived statistic reproduces exactly on its own construction: χ² = 41.13, df = 22, p = 0.0079, over 1,245 involvements in the thesis population.
- It comes from the **MallWorld** vertical report. The earlier `CLAUDE.md` withdrawal table listed it under the NDE entity-function claim, with "no source".
- Few cells are sparse (8%), but the involvements are nested: 535 dreams contribute, up to 16 involvements each. The χ² assumes independence it does not have.

**Corrected.** Over unique (dream, location, type) triples, with dream-clustered logits:

| Type | Underground | Ground | Elevated | Underground vs ground | Elevated vs ground |
|---|---|---|---|---|---|
| creature | 8.3% | 4.6% | 5.9% | OR 1.90 (1.00–3.61), p = 0.05 | OR 1.32, p = 0.42 |
| authority | 10.6% | 11.3% | 14.5% | OR 0.93, p = 0.78 | OR 1.33 (0.88–2.02), p = 0.18 |
| threat | 10.6% | 8.5% | 7.4% | OR 1.27, p = 0.42 | OR 0.86, p = 0.64 |

The archived notebook 09 called "creatures below, authority above" a pre-registered hit. That plan was written after the vertical report, which already contained this table.

**Verdict.**
- Creatures below: borderline (p = 0.05, exploratory).
- Authority above: **not supported**.

The pre-registered test of the specific framework prediction, that noxious animals appear below, was a **miss** (notebook 07, P2: OR 1.04). Whatever pulls creatures below is therefore not specific to harmful animals.

## E5 — Interactions: outcomes by entity demeanor, entity type and level

**Old.**
- Thesis §4.4: "Helpful entities → 49.7% success vs hostile 29.7%, χ² = 48.90".
- Entity ecology report: "friends 87.4%, guides 83.3%, family 70.3%"; "deceased: zero conflict across 56 encounters".
- Vertical report: "ground 68.3% vs elevated 63.2% vs underground 61.5% (χ² = 27.40, p = 0.007)".

In [6]:
DONE = ["succeeded", "failed", "interrupted", "abandoned"]
mw.check_values("interaction_outcome", DONE)
Io = I[I["interaction_outcome"].isin(DONE)].copy()
Io["success"] = (Io["interaction_outcome"] == "succeeded").astype(float)
print(f"Interactions with a definite outcome: {len(Io)} of {len(I)} ({100 * len(Io) / len(I):.1f}%), in {Io['post_id'].nunique()} dreams; overall success {fmt(Io, 'success')}")
Eo = EL[EL["interaction_outcome"].isin(DONE)].copy()
Eo["success"] = (Eo["interaction_outcome"] == "succeeded").astype(float)
for dem in [["helpful", "friendly"], ["hostile", "threatening"]]:
    print(f"  entity demeanor {dem}: {fmt(Eo[Eo['demeanor'].isin(dem)], 'success')}")
Eo["helpful_"] = Eo["demeanor"].isin(["helpful", "friendly"]).astype(int)
Eh = Eo[Eo["demeanor"].isin(["helpful", "friendly", "hostile", "threatening"])]
r = mw.clustered_logit(Eh, "success ~ helpful_ + log_words")
print("  success, helpful/friendly vs hostile/threatening:", mw.odds_ratio_table(r, ["helpful_"]).round(3).to_dict("index"))
for t in ["friend", "guide", "family_member", "authority", "threat", "deceased"]:
    print(f"  type {t:14s} {fmt(Eo[Eo['entity_type'] == t], 'success')}")
dec = EL[EL["entity_type"] == "deceased"]
nconf = int((dec["interaction_type"] == "conflict").sum())
nd = dec["post_id"].nunique()
print(f"\nDeceased: {len(dec)} involvements in {nd} dreams; conflict interactions {nconf}; upper 95% bound on the dream-level conflict rate (rule of three): {300 / nd:.1f}%")
print(f"Conflict share among all interactions: {100 * (I['interaction_type'] == 'conflict').mean():.1f}%")
dec_int = dec.drop_duplicates(["post_id", "interaction_index"])
base_conf = (I["interaction_type"] == "conflict").mean()
print(f"Distinct interactions involving a deceased person: {len(dec_int)}; expected conflicts at the base rate: {len(dec_int) * base_conf:.1f}; "
      f"P(0 conflicts | base rate) = {(1 - base_conf) ** len(dec_int):.3f}")
Il = Io.merge(L[["post_id", "location_id", "level_cat"]], on=["post_id", "location_id"], how="left").dropna(subset=["level_cat"])
print("\nSuccess by level:", {lvl: fmt(g, "success") for lvl, g in Il.groupby("level_cat", observed=True)})
Il["under"] = (Il["level_cat"] == "underground").astype(int); Il["elev"] = (Il["level_cat"] == "elevated").astype(int)
r = mw.clustered_logit(Il, "success ~ under + elev + log_words")
print("  vs ground:", mw.odds_ratio_table(r, ["under", "elev"]).round(3).to_dict("index"))
# Archived definitions: (a) thesis: all outcomes in the denominator; (b) entity-ecology report: succeeded / (succeeded + failed)
EL_all = EL.assign(success_all=(EL["interaction_outcome"] == "succeeded").astype(float))
print("\nThesis definition (all outcomes, incl. not_mentioned and ongoing, in the denominator):")
for dem in [["helpful"], ["hostile", "threatening"]]:
    print(f"  {dem}: {fmt(EL_all[EL_all['demeanor'].isin(dem)], 'success_all')}")
Esf = EL[EL["interaction_outcome"].isin(["succeeded", "failed"])].assign(s=lambda d: (d["interaction_outcome"] == "succeeded").astype(float))
print("Entity-ecology definition (succeeded / succeeded + failed):", {t: f"{100 * g['s'].mean():.1f}% (n = {len(g)})" for t, g in Esf.groupby("entity_type") if t in ["friend", "guide", "family_member", "deceased", "threat", "authority"]})
types_ok = Eo["entity_type"].value_counts()
Et = Eo[Eo["entity_type"].isin(types_ok[types_ok >= 20].index)]
r = mw.clustered_logit(Et, "success ~ C(entity_type) + log_words")
terms = [t for t in r.params.index if t.startswith("C(entity_type)")]
w = r.wald_test(" , ".join(f"{t} = 0" for t in terms), scalar=True)
print(f"Joint test that success differs by entity type ({len(terms) + 1} types with >= 20 outcomes; clustered, length-adjusted): chi2 = {float(w.statistic):.1f}, df = {len(terms)}, p = {float(w.pvalue):.3g}")

Interactions with a definite outcome: 2913 of 6056 (48.1%), in 1219 dreams; overall success 63.9% [61.8–65.9] (n = 2913, dreams = 1219)
  entity demeanor ['helpful', 'friendly']: 88.8% [85.1–92.5] (n = 303, dreams = 200)
  entity demeanor ['hostile', 'threatening']: 57.5% [51.6–63.3] (n = 362, dreams = 230)
  success, helpful/friendly vs hostile/threatening: {'helpful_': {'OR': 5.869, 'lo': 3.771, 'hi': 9.133, 'p': 0.0}}
  type friend         73.2% [53.9–92.4] (n = 41, dreams = 30)
  type guide          87.0% [75.4–98.5] (n = 23, dreams = 12)
  type family_member  61.1% [50.0–72.3] (n = 108, dreams = 72)
  type authority      57.7% [50.9–64.5] (n = 260, dreams = 172)
  type threat         60.0% [51.2–68.8] (n = 150, dreams = 98)
  type deceased       69.6% [54.0–85.1] (n = 23, dreams = 17)

Deceased: 56 involvements in 33 dreams; conflict interactions 0; upper 95% bound on the dream-level conflict rate (rule of three): 9.1%
Conflict share among all interactions: 5.6%
Distinct interacti


Success by level: {'underground': '64.1% [57.2–70.9] (n = 217, dreams = 130)', 'ground': '68.6% [64.1–73.1] (n = 506, dreams = 283)', 'elevated': '63.2% [57.4–69.1] (n = 302, dreams = 187)'}


  vs ground: {'under': {'OR': 0.817, 'lo': 0.569, 'hi': 1.174, 'p': 0.274}, 'elev': {'OR': 0.789, 'lo': 0.569, 'hi': 1.092, 'p': 0.153}}

Thesis definition (all outcomes, incl. not_mentioned and ongoing, in the denominator):
  ['helpful']: 63.3% [54.5–72.1] (n = 128, dreams = 99)
  ['hostile', 'threatening']: 28.1% [24.5–31.8] (n = 739, dreams = 416)
Entity-ecology definition (succeeded / succeeded + failed): {'authority': '72.5% (n = 207)', 'deceased': '72.7% (n = 22)', 'family_member': '74.2% (n = 89)', 'friend': '83.3% (n = 36)', 'guide': '95.2% (n = 21)', 'threat': '74.4% (n = 121)'}
Joint test that success differs by entity type (12 types with >= 20 outcomes; clustered, length-adjusted): chi2 = 31.6, df = 11, p = 0.000869


### Findings — E5

- **Outcome coverage.** An outcome (succeeded, failed, interrupted or abandoned) is coded for 48.1% of interactions.
- **Helpful vs hostile entities.**
  - Among definite outcomes, interactions with helpful or friendly entities succeed in 88.8% of cases, against 57.5% with hostile or threatening ones. This survives clustering and length adjustment: OR 5.87 (3.77–9.13).
  - The thesis's 49.7% vs 29.7% put "not mentioned" and "ongoing" in the denominator. It also counted a join that turned 7,283 interactions into 8,702 matches. Under the thesis's definition the primary population gives 63.3% vs 28.1%.
  - The direction is robust. This is not a framework test: help makes tasks succeed under any account.
- **Entity type.** Success differs by entity type: joint χ²(11) = 31.6, p = 0.0009, clustered and length-adjusted. The archived ranking does not reproduce. Under the entity-ecology report's definition (succeeded / (succeeded + failed)):
  - guide 95.2% (n = 21);
  - friend 83.3% (n = 36);
  - threat 74.4%;
  - family 74.2%;
  - deceased 72.7% (n = 22);
  - authority 72.5%.

  "Family members have the lowest success rate (70.3%)" is withdrawn.
- **Deceased: zero conflict.**
  - None of the 52 interactions that involve a deceased person is a conflict.
  - At the base rate of 5.6%, 2.9 conflicts would be expected, and the chance of seeing none is 0.050.
  - The direction fits the framework's account of meeting the departed (*Heaven and Hell* §494). The evidence is weak and was not specified in advance.
- **Level.**
  - Success is 64.1% underground, 68.6% at ground level and 63.2% elevated.
  - Against ground, the adjusted ORs are 0.82 (p = 0.27) underground and 0.79 (p = 0.15) elevated.
  - The archived χ² = 27.40 (df = 12) for outcome × level does not survive clustering and length adjustment. **No level effect.**

## E6 — Animal type and hostility (thesis §5.1, §5.3)

**Old.**
- "Animal type predicts demeanor, V = 0.513; predator 71% hostile vs cat 0% (OR = 106:1)".
- "Animal type clusters by dreamer, H = 70.91, η² = 0.605; ICC = 0.630".

Animal types were derived from text by keyword. The atmosphere claim (§5.2) came from the join bug and is replaced by pre-registered test P1 (notebook 07).

In [7]:
import re
# The archived keyword categories (notebook 09, cell 113), with its substring matching and priority rule
ARCHIVED = {
    "predator": ["wolf", "wolves", "lion", "tiger", "bear", "shark", "crocodile", "alligator", "predator", "hunting", "attack", "aggressive"],
    "serpent": ["snake", "serpent", "viper", "python", "boa", "reptile"],
    "bird": ["bird", "crow", "raven", "eagle", "hawk", "owl", "pigeon", "dove", "sparrow", "flying", "wings", "feather"],
    "insect": ["insect", "bug", "bee", "wasp", "spider", "fly", "flies", "ant", "roach", "cockroach", "mosquito", "yellow jacket", "moth", "beetle"],
    "dog": ["dog", "puppy", "canine", "retriever", "hound"],
    "cat": ["cat", "kitten", "feline"],
    "aquatic": ["fish", "whale", "dolphin", "otter", "aquatic", "swimming"],
    "monster": ["monster", "demon", "creature", "beast", "evil", "horror", "nightmare", "banshee", "ghoul", "shadow creature"],
    "domestic": ["rabbit", "guinea pig", "hamster", "cow", "horse", "deer", "sheep", "goat", "farm", "pet"],
}
BEHAVIOUR = {"hunting", "attack", "aggressive", "evil", "horror", "nightmare", "flying", "wings", "feather", "swimming",
             "farm", "pet", "creature", "beast", "aquatic", "reptile", "canine", "feline"}
PRIORITY = ["monster", "predator", "serpent", "insect", "bird", "aquatic", "dog", "cat", "domestic"]

def categorise(desc, whole_word=False, drop=frozenset()):
    if not isinstance(desc, str):
        return "unknown"
    d = desc.lower()
    hit = []
    for cat_, kws in ARCHIVED.items():
        for kw in kws:
            if kw in drop:
                continue
            found = re.search(rf"\b{re.escape(kw)}s?\b", d) if whole_word else (kw in d)
            if found:
                hit.append(cat_)
                break
    return next((p_ for p_ in PRIORITY if p_ in hit), "other")

cr = EL[EL["entity_type"] == "creature"].copy()
cr["archived"] = cr["entity_description"].map(categorise)
cr["nouns_only"] = cr["entity_description"].map(lambda d: categorise(d, whole_word=True, drop=BEHAVIOUR))
print(f"Creature involvements: {len(cr)} in {cr['post_id'].nunique()} dreams; hostile overall {fmt(cr, 'hostile_')}")
moved = cr[cr["archived"] != cr["nouns_only"]]
print(f"Classified differently once behaviour words and substring matches are removed: {len(moved)} of {len(cr)} ({100 * len(moved) / len(cr):.1f}%)")
print(pd.crosstab(cr["archived"], cr["nouns_only"]).to_string())
print("\nExamples that change category:")
for a_, b_, d in moved[["archived", "nouns_only", "entity_description"]].head(8).itertuples(index=False):
    print(f"  {a_} -> {b_}: {d[:100]}")

for col in ["archived", "nouns_only"]:
    tab = pd.crosstab(cr[col], cr["hostile_"])
    chi, pv, dof, _ = chi2_contingency(tab)
    print(f"\n{col}: chi2 = {chi:.1f}, df = {dof}, V = {mw.cramers_v(tab):.3f}; {mw.expected_count_report(tab)}")
    print(cr.groupby(col).agg(n=("hostile_", "size"), dreams=("post_id", "nunique"), hostile=("hostile_", "mean")).assign(hostile=lambda d: (100 * d["hostile"]).round(1)).sort_values("hostile", ascending=False).to_string())
attack_words = cr["entity_description"].fillna("").str.lower().str.contains("attack|aggressive|hunting|evil|horror|nightmare")
print(f"\nCreatures described with attack/aggressive/hunting/evil/horror/nightmare: {int(attack_words.sum())}; hostile {fmt(cr[attack_words], 'hostile_')} vs {fmt(cr[~attack_words], 'hostile_')} without")

Creature involvements: 194 in 130 dreams; hostile overall 39.2% [30.4–48.0] (n = 194, dreams = 130)
Classified differently once behaviour words and substring matches are removed: 39 of 194 (20.1%)
nouns_only  aquatic  bird  cat  dog  domestic  insect  monster  other  predator  serpent
archived                                                                                
aquatic          11     0    0    0         0       0        0      2         0        0
bird              0     3    0    0         0       0        0      1         0        0
cat               0     0   16    0         0       0        0      0         0        0
dog               0     0    0   19         0       0        0      0         0        0
domestic          0     0    0    0         8       0        0      0         0        0
insect            0     0    1    0         0      11        0      5         0        0
monster           1     2    0    1         0       1       18     18         0        0
ot


archived: chi2 = 48.4, df = 9, V = 0.500; min expected = 1.57; cells with expected < 5: 30%
           n  dreams  hostile
archived                     
predator  21      19     71.4
monster   41      32     63.4
bird       4       4     50.0
other     50      37     46.0
serpent    5       3     40.0
insect    17      15     29.4
domestic   8       7     12.5
aquatic   13      12      7.7
dog       19      15      5.3
cat       16      10      0.0

nouns_only: chi2 = 47.0, df = 9, V = 0.492; min expected = 1.96; cells with expected < 5: 35%
             n  dreams  hostile
nouns_only                     
monster     18      12     77.8
predator    14      12     64.3
bird         5       5     60.0
other       82      55     48.8
serpent      5       3     40.0
insect      13       9     38.5
domestic     8       7     12.5
aquatic     12      12      8.3
dog         20      16      5.0
cat         17      11      0.0

Creatures described with attack/aggressive/hunting/evil/horror/nigh

### Exploratory: the pre-registered animal classes and demeanor

Notebook 07 fixed whole-word lists of gentle and noxious animals before any test was run. Demeanor was not part of that registration, so this comparison is exploratory. It is also close to definitional: an animal described as fierce is one that is described as attacking.

In [8]:
GENTLE = set("cat cats kitten kittens kitty dog dogs puppy puppies sheep lamb lambs horse horses pony cow cows cattle calf deer fawn "
             "rabbit rabbits bunny bunnies bird birds dove doves chicken chickens duck ducks swan swans fish goldfish dolphin dolphins butterfly "
             "butterflies squirrel squirrels hamster turtle turtles owl owls".split())
NOXIOUS = set("snake snakes serpent serpents spider spiders rat rats mouse mice cockroach cockroaches roach roaches insect insects bug bugs "
              "ant ants wasp wasps hornet hornets scorpion scorpions worm worms maggot maggots leech leeches wolf wolves shark sharks crocodile "
              "crocodiles alligator alligators bear bears lion lions tiger tigers hound hounds hyena hyenas vulture vultures predator predators".split())
txt = (cr["entity_description"].fillna("") + " " + cr["name_or_relation"].fillna("")).str.lower()
toks = txt.str.replace(r"\bhot ?dogs?\b", " ", regex=True).str.findall(r"[a-z]+").map(set)
is_n = toks.map(lambda t: bool(t & NOXIOUS)); is_g = toks.map(lambda t: bool(t & GENTLE))
cr["cls"] = np.select([is_n & ~is_g, is_g & ~is_n], ["noxious", "gentle"], "unclassified")
for c_, g in cr.groupby("cls"):
    print(f"{c_:12s} hostile {fmt(g, 'hostile_')}")
cg = cr[cr["cls"] != "unclassified"].assign(nox=lambda d: (d["cls"] == "noxious").astype(int))
r = mw.clustered_logit(cg, "hostile_ ~ nox")
print("noxious vs gentle, clustered by dream:", mw.odds_ratio_table(r, ["nox"]).round(3).to_dict("index"))

gentle       hostile 11.5% [3.1–19.9] (n = 61, dreams = 46)
noxious      hostile 46.4% [23.3–69.5] (n = 28, dreams = 22)
unclassified hostile 53.3% [42.1–64.5] (n = 105, dreams = 76)


noxious vs gentle, clustered by dream: {'nox': {'OR': 6.686, 'lo': 1.977, 'hi': 22.615, 'p': 0.002}}


### "Animals cluster by dreamer" and "complete independence from atmosphere"

**Old.** ICC = 0.630 and Kruskal–Wallis H = 70.91 (η² = 0.605) for dreamer; atmosphere χ² ≈ 0, p = 1.0.

In [9]:
# Archived 'ICC' formula (notebook 09, cell 141): variance of author means over ALL authors / (that + mean within-author variance)
am = cr.groupby("author")["hostile_"].agg(["mean", "var", "size"])
bv, wv = am["mean"].var(), am["var"].dropna().mean()
print(f"Archived formula on the primary population: {bv / (bv + wv):.3f} (authors {len(am)}, of whom {int((am['size'] == 1).sum())} have one creature)")
multi = cr.groupby("author").filter(lambda g: len(g) >= 2)
one_dream = int(multi.groupby("author")["post_id"].nunique().eq(1).sum())
print(f"Authors with >= 2 creature involvements: {multi['author'].nunique()}; of these, all involvements in a single dream: {one_dream}")

# A proper author-level ICC: one score per dream, authors with >= 2 creature dreams
ds = cr.groupby("post_id").agg(score=("hostile_", "mean"), author=("author", "first"))
ds = ds[ds["author"].map(ds["author"].value_counts()) >= 2]
def icc1(df, group="author", y="score"):
    g = df.groupby(group)[y]; k = g.size(); n = len(df); a = len(k); grand = df[y].mean()
    msb = (k * (g.mean() - grand) ** 2).sum() / (a - 1)
    msw = ((df[y] - g.transform("mean")) ** 2).sum() / (n - a)
    k0 = (n - (k ** 2).sum() / n) / (a - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)
print(f"Dreams with creatures from authors with >= 2 such dreams: {len(ds)} ({ds['author'].nunique()} authors)")
if ds["author"].nunique() >= 3:
    obs = icc1(ds)
    rng = np.random.default_rng(3)
    perm = np.array([icc1(ds.assign(author=rng.permutation(ds["author"].values))) for _ in range(5000)])
    print(f"Dream-level ICC1 for creature hostility: {obs:.3f}; permutation p = {(1 + (perm >= obs).sum()) / 5001:.3f}")

# The archived atmosphere test
print("\nArchived test: dream means of the 1-5 atmosphere scale were cut at -0.5 and 0.5, so every dream fell in the 'positive' bin.")
print("The notebook printed 'Insufficient data for chi-square', set chi2 = 0 and p = 1, and the thesis reported this as 'complete independence'.")
cr_neg = cr.dropna(subset=["neg"])
print(f"Hostile creatures by location atmosphere (correct key): negative {fmt(cr_neg[cr_neg['neg'] == 1], 'hostile_')}; "
      f"non-negative {fmt(cr_neg[cr_neg['neg'] == 0], 'hostile_')}")
r = mw.clustered_logit(cr_neg, "hostile_ ~ neg")
print("  OR (negative vs non-negative atmosphere):", mw.odds_ratio_table(r, ["neg"]).round(3).to_dict("index"))

Archived formula on the primary population: 0.616 (authors 117, of whom 80 have one creature)
Authors with >= 2 creature involvements: 37; of these, all involvements in a single dream: 29
Dreams with creatures from authors with >= 2 such dreams: 21 (8 authors)


Dream-level ICC1 for creature hostility: 0.130; permutation p = 0.301

Archived test: dream means of the 1-5 atmosphere scale were cut at -0.5 and 0.5, so every dream fell in the 'positive' bin.
The notebook printed 'Insufficient data for chi-square', set chi2 = 0 and p = 1, and the thesis reported this as 'complete independence'.
Hostile creatures by location atmosphere (correct key): negative 59.0% [48.0–70.1] (n = 105, dreams = 74); non-negative 15.8% [0.0–37.7] (n = 19, dreams = 14)
  OR (negative vs non-negative atmosphere): {'neg': {'OR': 7.69, 'lo': 1.546, 'hi': 38.258, 'p': 0.013}}


### Findings — E6

**1. Animal type predicts demeanor: reproduced, and it does not depend on the circular keywords. It is also close to definitional.**
- The archived keyword scheme gives V = 0.500 on the primary population (χ²(9) = 48.4; 30% sparse cells). The archived value was 0.513.
- That scheme was circular and loose:
  - "predator" included *attack*, *aggressive* and *hunting*;
  - "monster" included *evil*, *horror* and *nightmare*;
  - matching was by substring.
- The 9 creatures described with those words are 100% hostile. Removing behaviour words and requiring whole words changes the category of 39 of 194 creatures (20.1%), yet still gives V = 0.492.
- "Predator vs cat OR = 106:1" rests on a zero cell (cats 0/17 hostile; predators 9/14). Such an OR is unbounded, so the number has no meaning as a magnitude.
- With the pre-registered word lists (exploratory for demeanor): noxious 46.4% vs gentle 11.5% hostile, OR 6.69 (1.98–22.6), clustered.
- **Interpretation.** Fierce animals are coded as hostile and tame ones as friendly. This fits *Heaven and Hell* §110, but any account predicts it.

**2. "Animals are completely independent of atmosphere (χ² ≈ 0, p = 1.0)": reversed.**
- The archived test cut dream means of the 1–5 atmosphere scale at −0.5 and 0.5, so every creature fell in the "positive" bin. The notebook printed "Insufficient data for chi-square" and then filled in χ² = 0, p = 1. The thesis reported that placeholder as "complete independence".
- With the correct key, creatures are hostile at 59.0% of negative locations, against 15.8% of non-negative ones: OR 7.69 (1.55–38.3), p = 0.013. Only 19 non-negative creature locations exist, in 14 dreams.
- The pre-registered P1 (notebook 07) confirms the corresponding framework prediction: noxious animals appear in negative scenes.
- Same-source caveat: a hostile animal in the narrative plausibly leads the coder to call the scene threatening.

**3. "Animals cluster by dreamer (ICC = 0.630, H = 70.91, η² = 0.605)": withdrawn.**
- The archived "ICC" divides the variance of author means, taken over all authors, by that variance plus the mean within-author variance. On the primary population this gives 0.616, but 80 of the 117 authors have one creature, so their means are 0 or 1. It is not an intraclass correlation.
- Of the 37 authors with at least two creatures, 29 have them all in one dream. The "dreamer" effect is mostly a within-dream effect.
- The archived notebook computed a proper dream-level ICC itself: 0.207 (95% CI 0.000–0.620), reported there as "consistency not established". The thesis omitted it.
- On the primary population: ICC1 = 0.130, permutation p = 0.30 (21 dreams from 8 authors). **Underdetermined.**
- The Kruskal–Wallis test on a binary outcome with groups of 2–5 has the same problem.

**Framework fidelity.** The thesis built its strongest claim on two bugs: animals are "independent of the environment" and "represent the dreamer". That reading also departs from the framework. In Swedenborg, animals that appear correspond to the affections of those present, and surroundings correspond to their state (*Heaven and Hell* §110, §§173–176). The framework therefore predicts that animals **co-vary** with the quality of the scene. The corrected data show exactly that co-variation: P1 was a hit, and the demeanor–atmosphere OR is 7.69. The dreamer-level claim cannot be evaluated with 8 authors.